我这里直接沿用whole_image里PCC后得到的features，然后去掉和shape相关的

In [1]:
# ============================================================
# Step 2: Habitat feature selection
# Use whole-tumor PCC-selected features, then remove shape features
# ============================================================

import os
import numpy as np
import pandas as pd

In [2]:
# ============================================================
# 1. Settings
# ============================================================

whole_pcc_path = "/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_PCC.xlsx"

habitat_root = "/host/d/projects/Habitats/radiomics/habitats"

habitat_normalized_path = os.path.join(
    habitat_root,
    "habitat_radiomics_measurements_avg_normalized.xlsx",
)

habitat_pcc_out_path = os.path.join(
    habitat_root,
    "habitat_radiomics_measurements_avg_PCC.xlsx",
)

feature_list_out_path = os.path.join(
    habitat_root,
    "habitat_PCC_feature_list_without_shape.xlsx",
)

non_feature_cols = [
    "Patient_set",
    "Patient_index",
    "Image_filepath",
    "Mask_filepath",
]

In [3]:
# ============================================================
# 2. Load whole-tumor PCC feature list
# ============================================================

whole_pcc_df = pd.read_excel(whole_pcc_path)

whole_pcc_features = [
    col for col in whole_pcc_df.columns
    if col not in non_feature_cols
]

shape_features = [
    col for col in whole_pcc_features
    if col.startswith("original_shape_")
]

final_features = [
    col for col in whole_pcc_features
    if col not in shape_features
]

print("Whole-tumor PCC table shape:", whole_pcc_df.shape)
print("Number of whole-tumor PCC features:", len(whole_pcc_features))
print("Number of shape features to remove:", len(shape_features))
print("Number of final habitat features:", len(final_features))

print("\nShape features removed:")
for f in shape_features:
    print(" ", f)

Whole-tumor PCC table shape: (330, 286)
Number of whole-tumor PCC features: 282
Number of shape features to remove: 7
Number of final habitat features: 275

Shape features removed:
  original_shape_Elongation
  original_shape_LeastAxisLength
  original_shape_MajorAxisLength
  original_shape_Maximum2DDiameterSlice
  original_shape_MeshVolume
  original_shape_Sphericity
  original_shape_SurfaceVolumeRatio


In [4]:
# ============================================================
# 3. Save final feature list
# ============================================================

feature_list_df = pd.DataFrame({
    "feature_name": final_features
})

removed_shape_df = pd.DataFrame({
    "removed_shape_feature": shape_features
})

with pd.ExcelWriter(feature_list_out_path, engine="openpyxl") as writer:
    feature_list_df.to_excel(writer, sheet_name="final_features", index=False)
    removed_shape_df.to_excel(writer, sheet_name="removed_shape_features", index=False)

print("Saved final feature list:", feature_list_out_path)

Saved final feature list: /host/d/projects/Habitats/radiomics/habitats/habitat_PCC_feature_list_without_shape.xlsx


In [6]:
# ============================================================
# 4. Load normalized weighted-average habitat radiomics
# ============================================================

habitat_df = pd.read_excel(habitat_normalized_path)

print("Habitat normalized table shape:", habitat_df.shape)

missing_non_feature_cols = [
    col for col in non_feature_cols
    if col not in habitat_df.columns
]

if len(missing_non_feature_cols) > 0:
    raise KeyError(f"Missing non-feature columns in habitat table: {missing_non_feature_cols}")

missing_features = [
    col for col in final_features
    if col not in habitat_df.columns
]

if len(missing_features) > 0:
    print("Missing selected features in habitat table:", len(missing_features))
    for f in missing_features:
        print(" ", f)

    raise KeyError("Some final selected features are missing from habitat normalized table.")

print("All final selected features are present in habitat table.")


# ============================================================
# 5. Apply feature selection to habitat table
# ============================================================

habitat_pcc_df = habitat_df[
    non_feature_cols + final_features
].copy()

print("Final habitat PCC table shape:", habitat_pcc_df.shape)

habitat_pcc_df.to_excel(
    habitat_pcc_out_path,
    index=False,
)

print("Saved:", habitat_pcc_out_path)

Habitat normalized table shape: (330, 1019)
All final selected features are present in habitat table.
Final habitat PCC table shape: (330, 279)
Saved: /host/d/projects/Habitats/radiomics/habitats/habitat_radiomics_measurements_avg_PCC.xlsx
